> **Interactive lesson:** Run the code cells, change inputs, and record your observations in the learner cells. This notebook is generated from the [Markdown source](03-tokenization.md); edit that source and rebuild rather than editing generated cells by hand.


# 1.3 — Tokenization

**Depth: LEARN**

**Goal:** understand how text becomes model-compatible IDs and why that mapping changes context capacity, cost, and behavior.

[Month 1 roadmap](../README.md) · [Previous: Position Information](02-position-information.ipynb) · [Next: LLM Training Objectives](04-llm-training-objectives.ipynb)


## Lesson 1.3.1 — The model reads IDs

A language model does not directly receive words or Unicode characters. A tokenizer transforms text into a sequence of vocabulary IDs. The embedding table maps each ID to a learned vector, and the LM head predicts a score for each possible next ID.

```text
text → normalization / pre-tokenization → pieces → vocabulary IDs
     → embedding lookup → transformer → vocabulary logits → ID → decoded text
```

Normalization and pre-tokenization are tokenizer-specific and may be absent or implemented differently. A token may represent a word, a fragment, punctuation, whitespace, several characters, or bytes. One visible character may consist of multiple Unicode code points; an emoji may use several UTF-8 bytes. Consequently, words, characters, bytes, and tokens are different accounting units.

A vocabulary ID is a categorical index. ID 200 is not “twice as meaningful” as ID 100. IDs are arbitrary within a fixed mapping, but changing their meaning after training breaks the embedding and output-head contract.


## Lesson 1.3.2 — Vocabulary construction and BPE

> **Read next:** [SentencePiece](https://arxiv.org/abs/1808.06226) explains language-independent subword training directly from raw text. The [Hugging Face tokenizers course chapter](https://huggingface.co/learn/llm-course/chapter6/1) adds practical inspection and training workflows.

Word-level vocabularies struggle with rare or novel words. Character/byte-level representations cover more input but produce longer sequences. Subword tokenizers trade vocabulary size against sequence length.

Byte-pair encoding (BPE) starts with small units and repeatedly merges a frequent adjacent pair in a training corpus. A toy symbol sequence might evolve as:

```text
l o w     → lo w → low
l o w e r → lo w e r → low e r
```

A production tokenizer records a learned vocabulary and merge ordering. Encoding new text applies those learned rules; it does not retrain the merge list on each prompt. Pre-tokenization rules can restrict where merges are allowed. Byte-level BPE starts from byte-covering units, which can represent unfamiliar text without a conventional unknown-word token, though it may use many tokens for that text.

Larger vocabularies can shorten sequences, but require larger embedding/output tables and more output logits. Efficiency is a system tradeoff, not simply “more tokens are bad” or “larger vocabulary is better.”


## Lesson 1.3.3 — WordPiece and SentencePiece

WordPiece also uses subword units. Its vocabulary-building criterion differs from ordinary frequency-based BPE; common encoders greedily choose the longest matching piece at each step. BERT-style vocabularies often mark continuation pieces with `##`. Depending on the tokenizer and its limits, an unsegmentable input can become an unknown token. Do not infer the entire algorithm from the visible piece prefix. [BERT](https://arxiv.org/abs/1810.04805) is a concrete historical example of WordPiece usage.

[SentencePiece](https://arxiv.org/abs/1808.06226) is a tokenizer training and encoding system, not a single subword algorithm. It supports approaches including BPE and unigram language modeling and can learn from raw sentences without requiring a language-specific word splitter. Its whitespace marker often makes spaces visible as part of pieces.

A unigram tokenizer begins with candidate pieces and optimizes a probabilistic vocabulary/segmentation model, removing less useful pieces. This differs from BPE's incremental pair merges. The practical questions are the actual model file, normalization policy, vocabulary, fallback behavior, and encode/decode contract.


## Lesson 1.3.4 — Special tokens and serialized conversations

Typical special-token roles include beginning of sequence, end of sequence, padding, unknown content, separators, and role boundaries. Names and IDs vary. A chat template converts roles and messages into the exact token sequence expected by a trained model; it is more than decorative text formatting.

Keep these distinctions clear:

- End-of-sequence is a learned generation symbol and may terminate decoding.
- Padding fills batches and normally needs attention/loss handling.
- An unknown token represents unsupported input according to the tokenizer's policy.
- A printable string that resembles a special token is not automatically a genuine role boundary. Tokenizer settings decide whether it is interpreted as a special ID or ordinary text.

Load the tokenizer and chat template associated with the checkpoint. A tensor containing only legal ID numbers can still be semantically wrong if it was created by a different tokenizer.


## Lesson 1.3.5 — A byte-level teaching baseline

For the project, a 256-entry UTF-8 byte tokenizer makes the text/ID relationship explicit. It is not BPE and will be less sequence-efficient than many subword tokenizers.


In [ ]:
text = "café"
ids = list(text.encode("utf-8"))
assert ids == [99, 97, 102, 195, 169]
assert bytes(ids).decode("utf-8") == text


There are four Unicode code points but five bytes/IDs here. Arbitrarily generated bytes may not form valid UTF-8. Decode generated samples with an explicit error policy, such as replacement characters, and record that limitation rather than claiming perfect text validity. The baseline has no special EOS token; stop using a maximum generation length unless you explicitly add and train an EOS convention.

A production subword tokenizer should be tested on leading spaces, newlines, punctuation, code, accented text, combining marks, CJK text, and emoji. For tokenizers with normalization, compare decoded text to the documented normalized form rather than assuming byte-identical round trips.


## Lesson 1.3.6 — Token efficiency affects architecture

Suppose two tokenizers encode the same document in 800 and 1,200 tokens. The second uses 1.5 times as many positions and KV-cache slots. Materialized full attention has 2.25 times as many pairwise coefficients. Actual latency also depends on projection cost, kernels, batching, and hardware.

If a service charges per token, the text's language and encoding affect cost. A fixed “four characters per token” estimate is a rough heuristic for some text, never a safe capacity check. Count with the actual tokenizer, including instructions, retrieved context, chat serialization, and output reservation.

For `D=512`, increasing a vocabulary from 16,000 to 32,000 adds about 8.2 million parameters to a single embedding table. Untied input/output weights add that amount twice. Weight tying shares their matrix, but the output distribution still has one score per vocabulary entry.

Tokenization also affects perplexity comparisons: a model predicting bytes and one predicting subwords solve different token-level prediction tasks. Compare losses only under compatible evaluation units, or convert to a common unit such as bits per byte with a carefully defined evaluation procedure.


## Checkpoint

1. Why must a tokenizer be versioned with the model checkpoint?
2. Is SentencePiece another name for BPE? Explain.
3. Why might two languages with equally long visible texts use different token budgets?
4. Is an EOS ID interchangeable with a PAD ID?


<details>
<summary>Show answers</summary>

1. Embedding rows and output logits have meanings tied to the vocabulary mapping and text preprocessing. Matching vocabulary sizes alone does not establish compatibility.
2. No. SentencePiece is a system supporting multiple algorithms, including BPE and unigram tokenization.
3. Their characters/bytes and learned subword coverage can differ. Normalization and pre-tokenization also matter; count actual token IDs.
4. No. EOS models an end event; padding is artificial batching content. Some deployments reuse an ID but still require carefully separate masking and stopping logic.

</details>


## Hands-on exercises

1. Using the byte baseline, compare the number of Python characters and tokens in `"hello"`, `"café"`, `"東京"`, and `"🙂"`. Verify round trips. Repeat later with the exact tokenizer of a pretrained model.
2. Design a tokenizer compatibility check for loading a model. Include more than the vocabulary size.
3. A tokenizer revision cuts sequence length from 2,000 to 1,500 for the same documents. Estimate relative KV-cache slots and full-attention coefficient counts. Identify a reason not to swap it into a pretrained model unchanged.


In [ ]:
# Your work here


<details>
<summary>Show exercise solutions</summary>

1. Character/byte-token counts are `5/5`, `4/5`, `2/6`, and `1/4`. These exact strings use precomposed `é`; a decomposed accent is a useful additional case.

    ```python
    for text in ["hello", "café", "東京", "🙂"]:
        ids = list(text.encode("utf-8"))
        assert bytes(ids).decode("utf-8") == text
        print(repr(text), len(text), len(ids))
    ```

2. Store a tokenizer artifact checksum/version, vocabulary mapping, special-token IDs, normalization rules, chat template, and expected encode/decode fixtures. Verify embedding/output dimensions and a few known serialized prompts. A matching dimension cannot catch a permuted ID mapping.
3. KV slots become `1500/2000 = 0.75` of baseline; coefficient counts become `0.75² = 0.5625`. The revised tokens may no longer correspond to the pretrained embedding/output rows, and sequence statistics may change; a tokenizer change needs model adaptation and evaluation.

</details>


## Completion criteria

Trace text to IDs and back, distinguish BPE/WordPiece/SentencePiece, explain vocabulary and sequence tradeoffs, and create a tokenizer compatibility audit.


## Primary references

- [SentencePiece](https://arxiv.org/abs/1808.06226) — system and supported tokenization approaches.
- [BERT](https://arxiv.org/abs/1810.04805) — WordPiece and special-token conventions in a specific model.
- [Neural Machine Translation of Rare Words with Subword Units](https://arxiv.org/abs/1508.07909) — BPE for subword modeling.


## Mapped companion lessons

- [Subword Tokenization](https://github.com/rohitg00/ai-engineering-from-scratch/tree/main/phases/05-nlp-foundations-to-advanced/19-subword-tokenization) compares BPE, WordPiece, Unigram, and SentencePiece.
- [Tokenizers](https://github.com/rohitg00/ai-engineering-from-scratch/tree/main/phases/10-llms-from-scratch/01-tokenizers) and [Building a Tokenizer from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch/tree/main/phases/10-llms-from-scratch/02-building-a-tokenizer) map to vocabulary construction and implementation.
- [BPE Tokenizer From Scratch](https://github.com/rohitg00/ai-engineering-from-scratch/tree/main/phases/19-capstone-projects/30-bpe-tokenizer-from-scratch) provides a focused capstone artifact.

See the [complete Month 0–1 content map](../../references/ai-engineering-from-scratch-map.md).
